In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

## Load and Prepare Data

In [2]:
# Load datasets
df_income = pd.read_csv('../data/raw/income_district.csv')
df_poverty = pd.read_csv('../data/raw/poverty_district.csv')
df_gini = pd.read_csv('../data/raw/gini_district.csv')
df_amenities = pd.read_csv('../data/raw/amenities.csv')

# Standardize column names
df_income.columns = df_income.columns.str.strip().str.lower()
df_poverty.columns = df_poverty.columns.str.strip().str.lower()
df_gini.columns = df_gini.columns.str.strip().str.lower()
df_amenities.columns = df_amenities.columns.str.strip().str.lower()

# Convert date to datetime and extract year
df_income['date'] = pd.to_datetime(df_income['date'])
df_income['year'] = df_income['date'].dt.year

df_poverty['date'] = pd.to_datetime(df_poverty['date'])
df_poverty['year'] = df_poverty['date'].dt.year

df_gini['date'] = pd.to_datetime(df_gini['date'])
df_gini['year'] = df_gini['date'].dt.year

df_amenities['date'] = pd.to_datetime(df_amenities['date'])
df_amenities['year'] = df_amenities['date'].dt.year

# Filter to years 2019 and 2022
df_income_filtered = df_income[df_income['year'].isin([2019, 2022])].copy()
df_poverty_filtered = df_poverty[df_poverty['year'].isin([2019, 2022])].copy()
df_gini_filtered = df_gini[df_gini['year'].isin([2019, 2022])].copy()
df_amenities_filtered = df_amenities[df_amenities['year'].isin([2019, 2022])].copy()

# Merge DataFrames
df_merged = df_income_filtered.copy()

df_merged = df_merged.merge(
    df_poverty_filtered[['state', 'district', 'year', 'poverty_absolute', 'poverty_relative']],
    on=['state', 'district', 'year'],
    how='left'
)

df_merged = df_merged.merge(
    df_gini_filtered[['state', 'district', 'year', 'gini']],
    on=['state', 'district', 'year'],
    how='left'
)

df_merged = df_merged.merge(
    df_amenities_filtered[['state', 'district', 'year', 'piped_water', 'sanitation', 'electricity']],
    on=['state', 'district', 'year'],
    how='left'
)

print(f"Merged DataFrame shape: {df_merged.shape}")

Merged DataFrame shape: (318, 12)


## Prepare Features and Target

In [3]:
# Define features and target
feature_cols = ['poverty_absolute', 'poverty_relative', 'gini', 'piped_water', 'sanitation', 'electricity']
target_col = 'income_median'

# Select features and target
X = df_merged[feature_cols]
y = df_merged[target_col]

print(f"Features shape before dropping missing: {X.shape}")
print(f"Target shape before dropping missing: {y.shape}")
print(f"\nMissing values in features:\n{X.isnull().sum()}")

Features shape before dropping missing: (318, 6)
Target shape before dropping missing: (318,)

Missing values in features:
poverty_absolute    0
poverty_relative    0
gini                0
piped_water         7
sanitation          6
electricity         6
dtype: int64


In [4]:
# Drop rows with missing feature values
valid_indices = X.dropna().index
X_clean = X.loc[valid_indices]
y_clean = y.loc[valid_indices]

print(f"\nFeatures shape after dropping missing: {X_clean.shape}")
print(f"Target shape after dropping missing: {y_clean.shape}")


Features shape after dropping missing: (311, 6)
Target shape after dropping missing: (311,)


## Train-Test Split

In [6]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 248
Test set size: 63


## Train Linear Regression Model

In [7]:
# Train linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


## Evaluate Model Performance

In [8]:
# Make predictions on test set
y_pred = model.predict(X_test)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Print results
print("Model Performance Metrics:")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.4f}")

Model Performance Metrics:
RMSE: 1350.52
MAE: 937.33
R²: 0.3301


## Cross-Validation Comparison

In [9]:
# Import required libraries for cross-validation
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Create a single KFold object to use for all models
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("KFold cross-validation setup complete (k=5, shuffle=True, random_state=42)")

KFold cross-validation setup complete (k=5, shuffle=True, random_state=42)


In [ ]:
# Define models to evaluate
models = {
    'Multiple Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
}

# Store results
results = []

# Evaluate each model using cross-validation
for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")
    
    # Get predictions using cross-validation
    y_pred_cv = cross_val_predict(model, X_clean, y_clean, cv=kf)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_clean, y_pred_cv))
    mae = mean_absolute_error(y_clean, y_pred_cv)
    r2 = r2_score(y_clean, y_pred_cv)
    
    # Store results
    results.append({
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    })
    
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  R²: {r2:.4f}")

# Create DataFrame with results
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("Cross-Validation Results Summary:")
print("="*60)
print(results_df.to_string(index=False))
print("="*60)


Evaluating Multiple Linear Regression...
  RMSE: 1245.54
  MAE: 896.69
  R²: 0.4019

Evaluating Ridge Regression...
  RMSE: 1238.13
  MAE: 888.13
  R²: 0.4090

Evaluating Random Forest Regressor...
  RMSE: 811.40
  MAE: 570.97
  R²: 0.7462

Cross-Validation Results Summary:
                     Model        RMSE        MAE       R2
Multiple Linear Regression 1245.544769 896.691767 0.401860
          Ridge Regression 1238.128206 888.127330 0.408962
   Random Forest Regressor  811.403417 570.974214 0.746161
